# A Minimal Many-Electron Wavefunction: Two Electrons, Spin, and the Slater Determinant

So far in this course we have met a single spin-½ particle (Computational Set 1), and we have learned to *couple* two-level systems together with tensor products (Computational Set 4, the two-qubit entangling notebook). This notebook puts those two skills together to build the smallest interesting model of a **many-electron wavefunction**: two electrons, described by their spin, assembled into a properly antisymmetric state — a **Slater determinant**.

There is one conceptual trap we need to walk around carefully, and it is worth stating up front because it shapes the whole notebook:

> Electrons are fermions, so the **total** (space $\times$ spin) wavefunction of a many-electron system must be **antisymmetric** under exchange of any two electrons. But this does **not** mean the spin part alone is antisymmetric, and it does **not** mean we abandon the ordinary tensor product when we couple spins.

To keep this straight we build the story in two clean layers:

- **Layer 1 — Coupling two spins.** The two-electron *spin* space is an ordinary Kronecker product $\mathbb{C}^2 \otimes \mathbb{C}^2$ — exactly the machinery from the two-qubit notebook. We build the total spin operators $\hat{S}_z$ and $\hat{S}^2$, and discover the **singlet** and **triplet** as their eigenstates. No antisymmetrization happens here.
- **Layer 2 — Antisymmetry and the Slater determinant.** Now we impose fermionic antisymmetry on the full **spin-orbital** wavefunction using the **wedge product** $\wedge$. We build a two-electron determinant, watch the Pauli principle appear for free, and connect the determinant picture back to the singlet and triplet of Layer 1.

We close with a short outlook to three and four electrons, and an aside connecting all of this to the second-quantized (ladder-operator) language.

Throughout we use atomic units with $\hbar = 1$, and represent kets as column vectors exactly as in the spin-½ notebook: $|\alpha\rangle = \begin{bmatrix}1\\0\end{bmatrix}$, $|\beta\rangle = \begin{bmatrix}0\\1\end{bmatrix}$. Following the vibrational-coupling chapter, we reserve $\hat{a},\hat{a}^\dagger$ / $\hat{b},\hat{b}^\dagger$ for ladder operators; here everything is spin, so no ladder operators appear until the closing aside.

## Layer 1 · Coupling Two Spins

### 1.1 One electron, then two

Recall the single spin-½ operators (with $\hbar = 1$):

$$
\hat{S}_x = \frac{1}{2}\begin{bmatrix}0&1\\1&0\end{bmatrix},\quad
\hat{S}_y = \frac{1}{2}\begin{bmatrix}0&-i\\i&0\end{bmatrix},\quad
\hat{S}_z = \frac{1}{2}\begin{bmatrix}1&0\\0&-1\end{bmatrix},\quad
\hat{S}^2 = \tfrac{3}{4}\,\mathbb{1}.
$$

The states $|\alpha\rangle$ and $|\beta\rangle$ are the eigenstates of $\hat{S}_z$ with eigenvalues $+\tfrac12$ and $-\tfrac12$, and both are eigenstates of $\hat{S}^2$ with eigenvalue $s(s+1) = \tfrac34$ (since $s = \tfrac12$).

For **two** electrons, the spin state lives in the tensor-product space $\mathbb{C}^2 \otimes \mathbb{C}^2$, spanned by four basis kets

$$
|\alpha\alpha\rangle,\quad |\alpha\beta\rangle,\quad |\beta\alpha\rangle,\quad |\beta\beta\rangle,
$$

built with `np.kron` exactly as you built composite states in the two-qubit notebook. **This is an ordinary Kronecker product — nothing is antisymmetrized yet.** The order of the two factors is a bookkeeping label for "electron 1" and "electron 2."

In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True, linewidth=120)

hbar = 1.0

# single-electron spin states (column vectors, as in the spin-1/2 notebook)
alpha = np.array([[1], [0]])
beta  = np.array([[0], [1]])

# single-electron spin operators
Sx = hbar/2 * np.array([[0, 1 ], [1,  0]])
Sy = hbar/2 * np.array([[0, -1j], [1j, 0]])
Sz = hbar/2 * np.array([[1, 0 ], [0, -1]])
S2_one = Sx @ Sx + Sy @ Sy + Sz @ Sz          # should be (3/4) I
I2 = np.eye(2)

print("S^2 for one electron (expect 0.75 on the diagonal):")
print(S2_one.real)

# two-electron spin basis via Kronecker products
aa = np.kron(alpha, alpha)   # |alpha alpha>
ab = np.kron(alpha, beta)    # |alpha beta>
ba = np.kron(beta,  alpha)   # |beta  alpha>
bb = np.kron(beta,  beta)    # |beta  beta>
print("\n|alpha beta> as a 4-vector:", ab.ravel())

S^2 for one electron (expect 0.75 on the diagonal):
[[0.75 0.  ]
 [0.   0.75]]

|alpha beta> as a 4-vector: [0 1 0 0]


### 1.2 Lifting operators to two particles

How does a spin operator act on the two-electron space? Through the **sum-over-particles** rule you already used for the two qubits: an operator for electron 1 acts on the first factor and leaves the second alone, and vice versa. For the total $z$-spin,

$$
\hat{S}_z^{\text{tot}} = \hat{S}_z \otimes \mathbb{1} + \mathbb{1} \otimes \hat{S}_z .
$$

The total $\hat{S}^2 = (\hat{\mathbf S}_1 + \hat{\mathbf S}_2)^2$ needs a little more care, because expanding the square produces a genuine **two-body** term:

$$
\hat{S}^2 = \hat{S}_1^2 + \hat{S}_2^2 + 2\,\hat{\mathbf S}_1\!\cdot\!\hat{\mathbf S}_2,
\qquad
\hat{\mathbf S}_1\!\cdot\!\hat{\mathbf S}_2 = \hat{S}_x\!\otimes\!\hat{S}_x + \hat{S}_y\!\otimes\!\hat{S}_y + \hat{S}_z\!\otimes\!\hat{S}_z .
$$

Two things are worth pausing on. First, this construction is built entirely from **tensor products and sums** — the same tools as the two-qubit notebook. **No wedge product appears when we lift operators.** (That is a common misconception: the wedge, coming in Layer 2, antisymmetrizes *states*, not operators.) Second, because $\hat{S}_z^{\text{tot}}$ and $\hat{S}^2$ commute with the operation that swaps the two electrons, they never mix the symmetric and antisymmetric parts of the space — a fact we will cash in shortly.

In [2]:
def lift_sum(op, dim=2):
    """Lift a one-particle operator to two particles by the sum-over-particles rule:
       op (x) I + I (x) op.   `dim` is the single-particle dimension."""
    I = np.eye(dim)
    return np.kron(op, I) + np.kron(I, op)

# total z-spin: a pure one-body (sum) operator
Sz_tot = lift_sum(Sz)

# total S^2 = S1^2 + S2^2 + 2 S1.S2   (the cross term is genuinely two-body)
S1_dot_S2 = np.kron(Sx, Sx) + np.kron(Sy, Sy) + np.kron(Sz, Sz)
S2_tot = np.kron(S2_one, I2) + np.kron(I2, S2_one) + 2 * S1_dot_S2

# both are valid observables -> Hermitian
assert np.allclose(Sz_tot, Sz_tot.conj().T)
assert np.allclose(S2_tot, S2_tot.conj().T)

print("Sz_tot (diagonal):", np.diag(Sz_tot).real)
print("\nS^2_tot:")
print(S2_tot.real)

Sz_tot (diagonal): [ 1.  0.  0. -1.]

S^2_tot:
[[2. 0. 0. 0.]
 [0. 1. 1. 0.]
 [0. 1. 1. 0.]
 [0. 0. 0. 2.]]


### 1.3 Which basis states are already "good" spin states?

A *good* spin state is a simultaneous eigenstate of $\hat{S}^2$ (total spin magnitude, quantum number $S$) and $\hat{S}_z^{\text{tot}}$ (its projection, quantum number $M_S$). Let us test each of our four Kronecker basis kets. We will need a small helper that decides whether a vector is an eigenstate of an operator and, if so, returns the eigenvalue.

In [3]:
def eigen_info(op, v, tol=1e-9):
    """If v is an eigenvector of op, return its eigenvalue; else return None."""
    w = op @ v
    nz = np.abs(v.ravel()) > tol
    ratios = w.ravel()[nz] / v.ravel()[nz]
    is_eig = np.allclose(ratios, ratios[0], atol=tol) and \
             np.allclose(w.ravel()[~nz], 0, atol=tol)
    return complex(ratios[0]) if is_eig else None

print(f"{'state':>14} | {'S^2 eigenvalue':>16} | {'Sz eigenvalue':>14}")
print("-" * 52)
for name, v in [("|aa>", aa), ("|ab>", ab), ("|ba>", ba), ("|bb>", bb)]:
    s2 = eigen_info(S2_tot, v)
    sz = eigen_info(Sz_tot, v)
    s2_str = f"{s2.real:+.2f}" if s2 is not None else "NOT an eigenstate"
    sz_str = f"{sz.real:+.2f}" if sz is not None else "NOT an eigenstate"
    print(f"{name:>14} | {s2_str:>16} | {sz_str:>14}")

print("\nWatch what S^2 does to |alpha beta>:")
print("S^2 |ab> =", (S2_tot @ ab).ravel().real, " = |ab> + |ba>  (a MIXTURE!)")

         state |   S^2 eigenvalue |  Sz eigenvalue
----------------------------------------------------
          |aa> |            +2.00 |          +1.00
          |ab> | NOT an eigenstate |          +0.00
          |ba> | NOT an eigenstate |          +0.00
          |bb> |            +2.00 |          -1.00

Watch what S^2 does to |alpha beta>:
S^2 |ab> = [0. 1. 1. 0.]  = |ab> + |ba>  (a MIXTURE!)


Two of the basis kets — $|\alpha\alpha\rangle$ and $|\beta\beta\rangle$ — are already good spin states. But $|\alpha\beta\rangle$ and $|\beta\alpha\rangle$, while perfectly good $\hat{S}_z$ eigenstates ($M_S = 0$), are **not** eigenstates of $\hat{S}^2$. Acting with $\hat{S}^2$ on $|\alpha\beta\rangle$ returns $|\alpha\beta\rangle + |\beta\alpha\rangle$ — a mixture of the two $M_S = 0$ kets.

This is a result students often find surprising: **$|\alpha\beta\rangle$ is not a singlet.** It is an equal mixture of the singlet and the $M_S = 0$ triplet. To get good spin states in the $M_S = 0$ block we must take particular linear combinations — the **spin-adapted** states.

### 1.4 The singlet and the triplet

The good spin states in the $M_S = 0$ block are the symmetric and antisymmetric combinations of $|\alpha\beta\rangle$ and $|\beta\alpha\rangle$. Together with the two "stretched" states, they organize into a **singlet** ($S = 0$, one state) and a **triplet** ($S = 1$, three states):

$$
\underbrace{\frac{|\alpha\beta\rangle - |\beta\alpha\rangle}{\sqrt2}}_{\text{singlet},\ S=0}
\qquad
\underbrace{|\alpha\alpha\rangle,\ \ \frac{|\alpha\beta\rangle + |\beta\alpha\rangle}{\sqrt2},\ \ |\beta\beta\rangle}_{\text{triplet},\ S=1,\ M_S = +1, 0, -1}.
$$

Let us build these four vectors and confirm their $S$ and $M_S$ quantum numbers directly.

In [4]:
singlet    = (ab - ba) / np.sqrt(2)
triplet_p1 = aa
triplet_0  = (ab + ba) / np.sqrt(2)
triplet_m1 = bb

def S_and_MS(v):
    s2 = eigen_info(S2_tot, v).real          # = S(S+1)
    ms = eigen_info(Sz_tot, v).real          # = M_S
    S  = (-1 + np.sqrt(1 + 4*s2)) / 2         # invert S(S+1)
    return S, ms

print(f"{'spin-adapted state':>22} | {'S':>4} | {'M_S':>5}")
print("-" * 40)
for name, v in [("singlet (ab - ba)", singlet),
                ("triplet  |aa>",     triplet_p1),
                ("triplet (ab + ba)", triplet_0),
                ("triplet  |bb>",     triplet_m1)]:
    S, ms = S_and_MS(v)
    print(f"{name:>22} | {S:>4.0f} | {ms:>+5.0f}")

# sanity: singlet has S=0, the three triplets share S=1
assert np.isclose(eigen_info(S2_tot, singlet).real, 0.0)
for v in (triplet_p1, triplet_0, triplet_m1):
    assert np.isclose(eigen_info(S2_tot, v).real, 2.0)   # S(S+1) = 2
print("\nAll spin quantum numbers confirmed.")

    spin-adapted state |    S |   M_S
----------------------------------------
     singlet (ab - ba) |    0 |    +0
         triplet  |aa> |    1 |    +1
     triplet (ab + ba) |    1 |    +0
         triplet  |bb> |    1 |    -1

All spin quantum numbers confirmed.


We can also let the computer *discover* this structure without our telling it the combinations, by diagonalizing $\hat{S}^2$. The eigenvalues should be $0$ (once) and $2$ (three times) — one singlet and a three-fold-degenerate triplet.

In [5]:
vals, vecs = np.linalg.eigh(S2_tot)
print("Eigenvalues of S^2_tot:", vals.real)
print("  -> one state at S(S+1)=0  (singlet)")
print("  -> three states at S(S+1)=2  (triplet, S=1)")

Eigenvalues of S^2_tot: [0. 2. 2. 2.]
  -> one state at S(S+1)=0  (singlet)
  -> three states at S(S+1)=2  (triplet, S=1)


### 1.5 Exchange symmetry: the seed of the Pauli story

Here is the bridge to Layer 2. Define the operator that **swaps the two electrons**, $\hat{P}_{12}\,|xy\rangle = |yx\rangle$. Apply it to each spin state and read off the sign:

In [6]:
# SWAP on C^2 (x) C^2: |xy> -> |yx>
SWAP = np.zeros((4, 4))
for i, ei in enumerate([alpha, beta]):
    for j, ej in enumerate([alpha, beta]):
        ket_ij = np.kron(ei, ej)
        ket_ji = np.kron(ej, ei)
        SWAP += ket_ji @ ket_ij.T

print(f"{'state':>22} | exchange P12 eigenvalue")
print("-" * 48)
for name, v in [("singlet (ab - ba)", singlet),
                ("triplet  |aa>",     triplet_p1),
                ("triplet (ab + ba)", triplet_0),
                ("triplet  |bb>",     triplet_m1)]:
    p = eigen_info(SWAP, v).real
    tag = "antisymmetric" if p < 0 else "symmetric"
    print(f"{name:>22} | {p:+.0f}   ({tag})")

                 state | exchange P12 eigenvalue
------------------------------------------------
     singlet (ab - ba) | -1   (antisymmetric)
         triplet  |aa> | +1   (symmetric)
     triplet (ab + ba) | +1   (symmetric)
         triplet  |bb> | +1   (symmetric)


The **singlet is antisymmetric** under exchange ($\hat{P}_{12} = -1$); the **three triplet states are symmetric** ($\hat{P}_{12} = +1$). This is the crucial fact that Layer 2 builds on. A two-electron wavefunction is a product of a spatial part and a spin part, and the *total* must be antisymmetric. So:

- symmetric spatial $\times$ **antisymmetric spin (singlet)** $\Rightarrow$ antisymmetric total ✓
- antisymmetric spatial $\times$ **symmetric spin (triplet)** $\Rightarrow$ antisymmetric total ✓

Notice we did **not** need to antisymmetrize anything to build the singlet and triplet — they simply emerged as eigenstates of $\hat{S}^2$ in the ordinary Kronecker space. Antisymmetry enters only when we insist on a full, physical fermionic wavefunction, which is Layer 2.

## Layer 2 · Antisymmetry and the Slater Determinant

### 2.1 The wedge product builds antisymmetric states

To describe a *real* two-electron wavefunction we must enforce antisymmetry on the **full single-particle states** — the **spin-orbitals**, each a combination of a spatial orbital and a spin function. The tool for this is the **wedge product** (also called the exterior or Grassmann product), written $\wedge$. For two single-particle states $|\phi\rangle$ and $|\chi\rangle$,

$$
|\phi\rangle \wedge |\chi\rangle \;=\; \frac{1}{\sqrt2}\Big(|\phi\rangle \otimes |\chi\rangle - |\chi\rangle \otimes |\phi\rangle\Big).
$$

A few things to appreciate before we code it:

- The wedge product **acts on ordinary state vectors** and returns an ordinary state vector — it is not something exotic reserved for density matrices. This antisymmetric vector *is* a **Slater determinant**: expanded in the product basis, its coefficients are literally the determinant of the coefficients of $|\phi\rangle$ and $|\chi\rangle$, which is where the name comes from.
- Swapping the two orbitals flips the sign: $|\chi\rangle \wedge |\phi\rangle = -\,|\phi\rangle \wedge |\chi\rangle$.
- Wedging a state with **itself** gives **zero** — this is the **Pauli exclusion principle**, appearing automatically.

(A side note to prevent a common mix-up: there is a *different* "wedge product of reduced density matrices" that shows up in advanced electronic-structure theory. That is a more specialized tool; the wedge here is the elementary exterior product of orbital vectors, and it is all we need.)

### 🧩 Design Recipe: `wedge2` — the two-electron antisymmetrizer

Following the recipe pattern from the two-qubit notebook:

1. **Header** `wedge2(phi, chi)` — two single-particle column vectors in, one antisymmetric two-particle vector out.
2. **Purpose** Return the normalized antisymmetric combination $\tfrac{1}{\sqrt2}(|\phi\rangle\otimes|\chi\rangle - |\chi\rangle\otimes|\phi\rangle)$, i.e. the Slater determinant of the two spin-orbitals.
3. **Body** One subtraction of two Kronecker products, divided by $\sqrt2$.
4. **Tests** Antisymmetry under swap, and the Pauli zero when the two arguments coincide.

In [7]:
def wedge2(phi, chi):
    """Two-electron Slater determinant of spin-orbitals |phi>, |chi>:
       (|phi>(x)|chi> - |chi>(x)|phi>) / sqrt(2)."""
    return (np.kron(phi, chi) - np.kron(chi, phi)) / np.sqrt(2)

# Test 1 - antisymmetry: swapping the orbitals flips the sign
assert np.allclose(wedge2(alpha, beta), -wedge2(beta, alpha))

# Test 2 - Pauli exclusion: identical spin-orbitals give the zero vector
assert np.allclose(wedge2(alpha, alpha), 0)

print("wedge2 passes antisymmetry and Pauli tests.")
print("\n|alpha> ^ |beta>  =", wedge2(alpha, beta).ravel(),
      "  <- this is exactly the singlet from Layer 1!")

wedge2 passes antisymmetry and Pauli tests.

|alpha> ^ |beta>  = [ 0.     0.707 -0.707  0.   ]   <- this is exactly the singlet from Layer 1!


Notice the punchline hiding in that last line: in a space with **only spin** (no spatial orbitals), the *only* antisymmetric two-electron state is $|\alpha\rangle \wedge |\beta\rangle$ — and it is precisely the **singlet**. There is no antisymmetric spin-only triplet, because the triplet spin states are symmetric. This is not an accident of our small basis; the antisymmetric subspace of $\mathbb{C}^2 \otimes \mathbb{C}^2$ is one-dimensional. To see a triplet as a determinant we must give the electrons somewhere to live in space.

### 2.2 Spin-orbitals: giving the electrons space

A **spin-orbital** is a spatial orbital times a spin function. The smallest model that lets us see both a singlet and a triplet uses **two spatial orbitals** $\{\phi_1, \phi_2\}$ (think $1s$ and $2s$) and the two spins $\{\alpha, \beta\}$, giving **four** spin-orbitals. We build each as a Kronecker product of a spatial ket and a spin ket, so the single-particle space is now $4$-dimensional (spatial $\otimes$ spin).

In [8]:
# spatial orbitals (a 2-dimensional toy spatial space): phi1 ~ "1s", phi2 ~ "2s"
phi1 = np.array([[1], [0]])
phi2 = np.array([[0], [1]])

# four spin-orbitals = spatial (x) spin   (each is a 4-vector)
phi1a = np.kron(phi1, alpha)   # phi1 with spin up
phi1b = np.kron(phi1, beta)    # phi1 with spin down
phi2a = np.kron(phi2, alpha)   # phi2 with spin up
phi2b = np.kron(phi2, beta)    # phi2 with spin down

print("spin-orbital phi1a as a 4-vector:", phi1a.ravel())
print("single-particle spin-orbital space is now 4-dimensional")

spin-orbital phi1a as a 4-vector: [1 0 0 0]
single-particle spin-orbital space is now 4-dimensional


To assign spin quantum numbers to these two-electron determinants we need $\hat{S}_z^{\text{tot}}$ and $\hat{S}^2$ **in the spin-orbital space**. The lifting rule is unchanged — sum over particles — but now each single-particle spin operator must act only on the spin half of a spin-orbital, so we pad it with the spatial identity: $\hat{S}_z^{\text{so}} = \mathbb{1}_{\text{spatial}} \otimes \hat{S}_z$.

In [9]:
# spin operators acting on a single spin-orbital act on the SPIN part only
Sx_so = np.kron(I2, Sx)
Sy_so = np.kron(I2, Sy)
Sz_so = np.kron(I2, Sz)
S2_so_one = Sx_so @ Sx_so + Sy_so @ Sy_so + Sz_so @ Sz_so   # (3/4) I_4
I4 = np.eye(4)

# two-electron totals in the 16-dim spin-orbital space
Sz_tot_so = np.kron(Sz_so, I4) + np.kron(I4, Sz_so)
S1S2_so   = np.kron(Sx_so, Sx_so) + np.kron(Sy_so, Sy_so) + np.kron(Sz_so, Sz_so)
S2_tot_so = np.kron(S2_so_one, I4) + np.kron(I4, S2_so_one) + 2 * S1S2_so

print("Spin operators lifted into the 16-dimensional two-electron spin-orbital space.")

Spin operators lifted into the 16-dimensional two-electron spin-orbital space.


### 2.3 The closed-shell ground state is a single determinant

Put both electrons in the lowest spatial orbital $\phi_1$, with opposite spins. The Slater determinant is $|\phi_1\alpha\rangle \wedge |\phi_1\beta\rangle$. This is the model of a closed-shell ground state (think of helium's $1s^2$). Let us build it and read off its spin.

In [10]:
He = wedge2(phi1a, phi1b)          # closed-shell "1s^2" determinant

S2 = eigen_info(S2_tot_so, He).real
Sz = eigen_info(Sz_tot_so, He).real
print(f"closed-shell determinant  |phi1a ^ phi1b>:   S(S+1) = {S2:.1f},  M_S = {Sz:+.1f}")
print("  -> S = 0, M_S = 0: a SINGLET.")

# WHY is it a singlet? Reshape the 16-vector as (orb1, spin1, orb2, spin2)
# and inspect the surviving amplitudes.
T = He.reshape(2, 2, 2, 2)
labels = {0: "phi1", 1: "phi2"}, {0: "alpha", 1: "beta"}
print("\nnonzero (orbital1, spin1, orbital2, spin2) amplitudes:")
for idx in np.argwhere(np.abs(T) > 1e-9):
    o1, s1, o2, s2 = idx
    print(f"  ({labels[0][o1]}, {labels[1][s1]}, {labels[0][o2]}, {labels[1][s2]}) "
          f"-> {T[o1, s1, o2, s2]:+.3f}")
print("\nBoth electrons occupy phi1 (SYMMETRIC in space); the spin part is")
print("(alpha beta - beta alpha)/sqrt(2) = the SINGLET.  Closed shells are singlets.")

closed-shell determinant  |phi1a ^ phi1b>:   S(S+1) = 0.0,  M_S = +0.0
  -> S = 0, M_S = 0: a SINGLET.

nonzero (orbital1, spin1, orbital2, spin2) amplitudes:
  (phi1, alpha, phi1, beta) -> +0.707
  (phi1, beta, phi1, alpha) -> -0.707

Both electrons occupy phi1 (SYMMETRIC in space); the spin part is
(alpha beta - beta alpha)/sqrt(2) = the SINGLET.  Closed shells are singlets.


The determinant cleanly factors into a **symmetric spatial part** (both electrons in $\phi_1$) times an **antisymmetric spin part** (the singlet) — exactly the pairing anticipated at the end of Layer 1. And if we try to put both electrons in the *same* spin-orbital, the Pauli zero returns:

In [11]:
print("Attempt to put two electrons in the identical spin-orbital phi1a:")
print("  |phi1a ^ phi1a> =", wedge2(phi1a, phi1a).ravel())
print("  -> the zero vector: two electrons cannot share a spin-orbital (Pauli).")

Attempt to put two electrons in the identical spin-orbital phi1a:
  |phi1a ^ phi1a> = [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  -> the zero vector: two electrons cannot share a spin-orbital (Pauli).


### 2.4 The triplet needs two orbitals — and spin-adaptation returns

To build a triplet we must occupy **two different spatial orbitals**. The stretched ($M_S = \pm1$) triplet components are still single determinants — for example $|\phi_1\alpha\rangle \wedge |\phi_2\alpha\rangle$ has both spins up. But the $M_S = 0$ states are subtler: neither single $M_S = 0$ determinant is a spin eigenstate on its own. We must take **linear combinations of determinants** to recover good spin states. This is exactly the **spin-adaptation** of Layer 1, now living in the determinant picture.

In [12]:
# M_S = +1 triplet: a single determinant (both spins up, different orbitals)
trip_p1 = wedge2(phi1a, phi2a)

# the two M_S = 0 determinants
detA = wedge2(phi1a, phi2b)        # |phi1(up) phi2(down)|
detB = wedge2(phi1b, phi2a)        # |phi1(down) phi2(up)|

# spin-adapted combinations
open_singlet = (detA - detB) / np.sqrt(2)
trip_0       = (detA + detB) / np.sqrt(2)

def report(name, v):
    s2 = eigen_info(S2_tot_so, v)
    sz = eigen_info(Sz_tot_so, v).real
    if s2 is None:
        print(f"{name:>26}:  NOT an S^2 eigenstate   (M_S = {sz:+.0f})")
    else:
        S = (-1 + np.sqrt(1 + 4*s2.real)) / 2
        print(f"{name:>26}:  S = {S:.0f}, M_S = {sz:+.0f}")

report("triplet +1  |1up ^ 2up|", trip_p1)
report("detA alone |1up 2down|", detA)
report("detB alone |1down 2up|", detB)
report("(detA - detB)/sqrt2", open_singlet)
report("(detA + detB)/sqrt2", trip_0)

assert eigen_info(S2_tot_so, detA) is None            # single det: not spin-pure
assert np.isclose(eigen_info(S2_tot_so, open_singlet).real, 0.0)   # open-shell singlet
assert np.isclose(eigen_info(S2_tot_so, trip_0).real,       2.0)   # M_S=0 triplet
print("\nSingle M_S=0 determinants are NOT spin eigenstates; their +/- combinations are.")

   triplet +1  |1up ^ 2up|:  S = 1, M_S = +1
    detA alone |1up 2down|:  NOT an S^2 eigenstate   (M_S = +0)
    detB alone |1down 2up|:  NOT an S^2 eigenstate   (M_S = +0)
       (detA - detB)/sqrt2:  S = 0, M_S = +0
       (detA + detB)/sqrt2:  S = 1, M_S = +0

Single M_S=0 determinants are NOT spin eigenstates; their +/- combinations are.


This is one of the most important lessons in the whole notebook, and it is easy to state:

> A single Slater determinant is **not always** an eigenstate of $\hat{S}^2$. The stretched triplet components happen to be single determinants, but the $M_S = 0$ triplet and the open-shell singlet are each an equal-weight combination of **two** determinants. States built to be proper spin eigenstates are called **configuration state functions (CSFs)**, and this is why quantum-chemistry methods often work with spin-adapted combinations rather than raw determinants.

The circle is now closed: the singlet and triplet that fell out of the ordinary Kronecker coupling in Layer 1 are exactly the spin structures carried by the antisymmetric determinants of Layer 2.

## Beyond Two Electrons (Outlook)

Everything generalizes. For $N$ electrons the Slater determinant is the antisymmetrized product of $N$ spin-orbitals, written as a signed sum over all permutations $\sigma$ of the labels:

$$
|\phi_1 \wedge \phi_2 \wedge \cdots \wedge \phi_N\rangle
= \frac{1}{\sqrt{N!}} \sum_{\sigma \in S_N} \operatorname{sgn}(\sigma)\,
|\phi_{\sigma(1)}\rangle \otimes |\phi_{\sigma(2)}\rangle \otimes \cdots \otimes |\phi_{\sigma(N)}\rangle .
$$

The function below implements exactly this. It reduces to `wedge2` for $N = 2$, and the Pauli principle persists: repeat any spin-orbital and the determinant collapses to zero.

In [13]:
from itertools import permutations
from math import factorial

def parity(perm):
    """Sign (+1/-1) of a permutation given as a tuple of 0..N-1."""
    perm = list(perm); sign = 1
    for i in range(len(perm)):
        while perm[i] != i:
            j = perm[i]
            perm[i], perm[j] = perm[j], perm[i]
            sign = -sign
    return sign

def slater(*orbitals):
    """General N-electron Slater determinant of the given spin-orbital vectors."""
    N = len(orbitals)
    out = None
    for sigma in permutations(range(N)):
        term = orbitals[sigma[0]]
        for k in sigma[1:]:
            term = np.kron(term, orbitals[k])
        out = parity(sigma) * term if out is None else out + parity(sigma) * term
    return out / np.sqrt(factorial(N))

# reduces to wedge2 for N=2
assert np.allclose(slater(phi1a, phi1b), wedge2(phi1a, phi1b))

# a 3-electron determinant is nonzero when all spin-orbitals differ ...
three = slater(phi1a, phi1b, phi2a)
print("3-electron determinant is nonzero:", not np.allclose(three, 0),
      " (dimension", three.shape[0], "= 4^3)")

# ... and zero the moment two spin-orbitals coincide (Pauli)
assert np.allclose(slater(phi1a, phi1b, phi1a), 0)
print("Repeating a spin-orbital gives zero, for any N (Pauli).")

3-electron determinant is nonzero: True  (dimension 64 = 4^3)
Repeating a spin-orbital gives zero, for any N (Pauli).


Two honest caveats to carry into the three- and four-electron cases. First, the full tensor space has dimension $d^N$ (with $d$ the single-particle dimension), so these explicit constructions are teaching tools, not production methods. Second, a spin-**only** space caps at two electrons — you cannot antisymmetrize three vectors in a two-dimensional space — so any $N \ge 3$ example *requires* at least $N$ spatial orbitals. Building the $N = 3$ and $N = 4$ determinants, and spin-adapting them into CSFs, is a natural next notebook.

### 🌉 A grown-up view: second quantization

The wedge product does its antisymmetry bookkeeping *by hand* — every determinant is a signed sum over permutations, and we track those signs ourselves. The professional tool that automates all of it is **second quantization**, the same ladder-operator language from the harmonic-oscillator / cavity notebook, now applied to electrons.

There we introduce a **fermionic creation operator** $\hat{c}_p^\dagger$ that adds an electron to spin-orbital $p$, and its adjoint $\hat{c}_p$ that removes one. The single defining property,

$$
\{\hat{c}_p, \hat{c}_q^\dagger\} \equiv \hat{c}_p\hat{c}_q^\dagger + \hat{c}_q^\dagger\hat{c}_p = \delta_{pq},
\qquad
\{\hat{c}_p^\dagger, \hat{c}_q^\dagger\} = 0,
$$

is the *anticommutator* (contrast the *commutator* $[\hat{b}, \hat{b}^\dagger] = 1$ for photons). A Slater determinant is then simply a string of creation operators acting on the empty state (the *vacuum* $|\text{vac}\rangle$):

$$
|\phi_1 \wedge \cdots \wedge \phi_N\rangle \;\longleftrightarrow\; \hat{c}_1^\dagger \hat{c}_2^\dagger \cdots \hat{c}_N^\dagger\,|\text{vac}\rangle .
$$

The two Pauli facts we verified by hand now come for free from the algebra: $\hat{c}_p^\dagger \hat{c}_p^\dagger = 0$ (can't occupy a spin-orbital twice), and $\hat{c}_p^\dagger \hat{c}_q^\dagger = -\hat{c}_q^\dagger \hat{c}_p^\dagger$ (swapping two orbitals flips the sign). It is the fermionic cousin of the bosonic ladder operators you already know — same idea, one crucial sign.

### Try it yourself

1. **Symmetry census.** Apply the `SWAP` operator to all four Layer-1 spin states and confirm the singlet is the lone antisymmetric one. (We did this above — reproduce it from scratch to check your understanding.)
2. **Match the two layers.** Show numerically that the spin part of the $M_S = 0$ determinant combination `trip_0` from Layer 2 equals the Layer-1 triplet spin function $(|\alpha\beta\rangle + |\beta\alpha\rangle)/\sqrt2$.
3. **A different closed shell.** Build the determinant with both electrons in $\phi_2$ instead of $\phi_1$ and confirm it is also a singlet.
4. **Three electrons.** Using `slater`, build a three-electron determinant from $\{\phi_1\alpha, \phi_1\beta, \phi_2\alpha\}$, then compute its $M_S$ using the three-particle $\hat{S}_z^{\text{tot}}$ (you will need to lift $\hat{S}_z^{\text{so}}$ to three particles). What total $M_S$ do you expect?
5. **Pauli hunt.** Confirm that *any* repeated spin-orbital — not just the first two slots — sends `slater` to zero.

### That's a wrap!

In this notebook we:

- coupled two spins in the ordinary Kronecker space $\mathbb{C}^2 \otimes \mathbb{C}^2$, lifting $\hat{S}_z$ and $\hat{S}^2$ with the **same sum-over-particles machinery** as the two-qubit notebook — **no antisymmetrization required** — and found the **singlet** and **triplet** as $\hat{S}^2$ eigenstates;
- saw that $|\alpha\beta\rangle$ is **not** a singlet but a mixture, so good spin states demand **spin-adapted** combinations;
- traced the **exchange symmetry** of each spin state: the singlet is antisymmetric, the triplet symmetric;
- introduced the **wedge product** as an operation on ordinary **state vectors** that builds a **Slater determinant**, with the **Pauli principle** ($|\phi\rangle \wedge |\phi\rangle = 0$) appearing automatically;
- built the **closed-shell** determinant and showed it is symmetric-in-space $\times$ singlet-in-spin, and showed a triplet requires **two spatial orbitals**;
- discovered that some spin eigenstates (the $M_S=0$ triplet, the open-shell singlet) are **combinations of determinants** — the origin of **configuration state functions**;
- and connected the whole picture to **second quantization**, where fermionic anticommutation automates the sign bookkeeping.

Two ideas to carry forward: **operators are lifted by sums over particles; states are antisymmetrized by wedges** — never the other way around; and a **single determinant is not always spin-pure**.

### Read these next

- Computational Set 1 (spin-½) and Computational Set 4 (two-qubit coupling), whose machinery this notebook extends, and the second-quantization / cavity notebook for the ladder-operator language behind the closing aside.
- A. Szabo and N. S. Ostlund, *Modern Quantum Chemistry* (Dover, 1996) — the standard, readable introduction to Slater determinants, spin-adaptation, and the Hartree–Fock method.
- T. Helgaker, P. Jørgensen, and J. Olsen, *Molecular Electronic-Structure Theory* (Wiley, 2000) — the comprehensive reference, including the second-quantized formulation.